# **Text Generation with Recurrent Neural Networks (RNN)**

**1. Import Necessary Libraries**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import requests
from bs4 import BeautifulSoup
import re
import numpy as np
import collections
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Setting random seeds
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda


**2. Web Scraping the Data**

In [2]:
url = "https://www.gutenberg.org/cache/epub/84/pg84-images.html"
print("Downloading text...")
response = requests.get(url)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, 'html.parser')
raw_text = soup.get_text()

start_marker = "Letter 1"
end_marker = "THE END"

try:
    start_idx = raw_text.find(start_marker)
    end_idx = raw_text.find(end_marker)
    if start_idx != -1 and end_idx != -1:
        text_data = raw_text[start_idx:end_idx + len(end_marker)]
    else:
        text_data = raw_text
except:
    text_data = raw_text

print(f"Text length: {len(text_data)} characters.")

Text length: 446307 characters.


**3. Data Preprocessing and Tokenization**

In [3]:
text_data = re.sub(r'([.!?])', r'\1 <END>', text_data)

tokens = re.findall(r"\w+|[^\w\s]|<END>", text_data.lower(), re.UNICODE)
vocab = sorted(list(set(tokens)))
vocab_size = len(vocab)

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}

print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 7336


**4. Preparing Sequences for RNN**

In [4]:
window_size = 100
seq_length = window_size - 1
inputs = []
targets = []
step = 1

token_indices = [word_to_idx[word] for word in tokens]

for i in range(0, len(token_indices) - window_size, step):
    inputs.append(token_indices[i : i + seq_length])
    targets.append(token_indices[i + seq_length])

print(f"Number of sequences: {len(inputs)}")

Number of sequences: 100382


**5. Custom Dataset and DataLoader**

In [5]:
class RNNTextDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = torch.LongTensor(inputs)
        self.targets = torch.LongTensor(targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

BATCH_SIZE = 64
dataset = RNNTextDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("DataLoader ready.")

DataLoader ready.


**6. RNN Model Implementation**

In [6]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers=2, dropout=0.3):
        super(SimpleRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            embedding_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)  # ուղղակի hidden → vocab

        total_params = sum(p.numel() for p in self.parameters())
        print(f"[MODEL] Params: {total_params:,} | LayerNorm: ✓")

    def forward(self, x, hidden):
        embeds = self.dropout(self.embedding(x))
        out, hidden = self.rnn(embeds, hidden)
        out = self.layer_norm(out[:, -1, :])
        out = self.dropout(out)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)

**7. Hyperparameters and Model Initialization**

In [7]:
EMBEDDING_DIM  = 256
HIDDEN_DIM     = 512
N_LAYERS       = 2
DROPOUT        = 0.3
LEARNING_RATE  = 0.001
EPOCHS         = 30

model = SimpleRNN(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).to(device)

def init_weights(m):
    if type(m) == nn.RNN:
        for name, param in m.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
                print(f"  [INIT] Orthogonal → {name}")
    elif type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
        print(f"  [INIT] Xavier     → Linear layer")

print("[WEIGHT INIT]")
model.apply(init_weights)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5
)


print(f"\n[HYPERPARAMS]")
print(f"  Learning Rate : {LEARNING_RATE}")
print(f"  Epochs        : {EPOCHS}")
print(f"  Batch Size    : {BATCH_SIZE}")
print(f"  Device        : {device}")

[MODEL] Params: 6,561,960 | LayerNorm: ✓
[WEIGHT INIT]
  [INIT] Orthogonal → weight_ih_l0
  [INIT] Orthogonal → weight_hh_l0
  [INIT] Orthogonal → weight_ih_l1
  [INIT] Orthogonal → weight_hh_l1
  [INIT] Xavier     → Linear layer

[HYPERPARAMS]
  Learning Rate : 0.001
  Epochs        : 30
  Batch Size    : 64
  Device        : cuda


**8. Training Loop**

In [8]:
def generate_text_gpu(model, seed_text, next_words=100, temperature=1.0, rep_penalty=1.3, greedy=False):
    model.eval()
    words = re.findall(r"\w+|[^\w\s]|<END>", seed_text.lower(), re.UNICODE)
    current_seq = [word_to_idx.get(w, 0) for w in words]
    x_in = torch.LongTensor([current_seq]).to(device)
    hidden = model.init_hidden(1)
    generated = words[:]

    with torch.no_grad():
        _, hidden = model(x_in, hidden)
        last_word_idx = current_seq[-1]

        for _ in range(next_words):
            x_in = torch.LongTensor([[last_word_idx]]).to(device)
            output, hidden = model(x_in, hidden)
            logits = output[0] / temperature

            recent_ids = [word_to_idx.get(w, 0) for w in generated[-20:]]
            for prev_id in set(recent_ids):
                logits[prev_id] /= rep_penalty

            if greedy:
                pred_idx = torch.argmax(logits).item()
            else:
                probs = torch.softmax(logits, dim=0).cpu().numpy()
                pred_idx = np.random.choice(len(probs), p=probs)

            pred_word = idx_to_word[pred_idx]

            if pred_word == "<end>":
                generated.append(".")
                break

            generated.append(pred_word)
            last_word_idx = pred_idx

    model.train()
    return " ".join(generated)

# ---- Seed text for samples ----
start_index = random.randint(0, len(tokens) - 150)
seed_text = " ".join(tokens[start_index: start_index + 20])

# ---- Then training loop ----
import time
import math

print(f"{'='*55}")
print(f"  Training started on {device}")
print(f"{'='*55}\n")

best_loss = float('inf')
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start_time = time.time()

    for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        hidden = model.init_hidden(BATCH_SIZE)
        hidden = hidden.detach()

        optimizer.zero_grad()

        output, hidden = model(x_batch, hidden)
        loss = criterion(output, y_batch)

        loss.backward()

        # Gradient clipping — 5 → 1 (KEY FIX)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        # Batch progress — every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"  [Epoch {epoch+1:02d} | Batch {batch_idx+1:3d}/{len(dataloader)}] "
                  f"Loss: {loss.item():.4f} | GradNorm: {grad_norm:.3f}")

    avg_loss = total_loss / len(dataloader)
    perplexity = math.exp(avg_loss)
    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr']

    loss_history.append(avg_loss)
    scheduler.step()

    # Best model saving
    improved = ""
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), 'best_rnn_model.pt')
        improved = "  ✓ Best model saved!"

    print(f"\nEpoch {epoch+1:02d}/{EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Perplexity: {perplexity:.1f} | "
          f"LR: {current_lr:.6f} | "
          f"Time: {elapsed:.1f}s"
          f"{improved}\n")

    if (epoch + 1) % 5 == 0:
        print(f"  [SAMPLE at epoch {epoch+1}]")
        sample = generate_text_gpu(model, seed_text, next_words=30, temperature=0.8)
        sample_words = sample.split()
        print("  " + " ".join(sample_words[20:50]))
        print()

print(f"{'='*55}")
print(f"  Training finished! Best loss: {best_loss:.4f}")
print(f"{'='*55}")

  Training started on cuda

  [Epoch 01 | Batch  50/1568] Loss: 5.8146 | GradNorm: 3.785
  [Epoch 01 | Batch 100/1568] Loss: 6.1052 | GradNorm: 3.959
  [Epoch 01 | Batch 150/1568] Loss: 6.3345 | GradNorm: 3.711
  [Epoch 01 | Batch 200/1568] Loss: 5.0705 | GradNorm: 3.332
  [Epoch 01 | Batch 250/1568] Loss: 6.3667 | GradNorm: 3.593
  [Epoch 01 | Batch 300/1568] Loss: 6.0136 | GradNorm: 3.241
  [Epoch 01 | Batch 350/1568] Loss: 5.6223 | GradNorm: 3.351
  [Epoch 01 | Batch 400/1568] Loss: 5.4711 | GradNorm: 3.328
  [Epoch 01 | Batch 450/1568] Loss: 5.8471 | GradNorm: 3.182
  [Epoch 01 | Batch 500/1568] Loss: 5.0928 | GradNorm: 3.122
  [Epoch 01 | Batch 550/1568] Loss: 5.4585 | GradNorm: 3.113
  [Epoch 01 | Batch 600/1568] Loss: 5.4452 | GradNorm: 3.161
  [Epoch 01 | Batch 650/1568] Loss: 5.4062 | GradNorm: 2.932
  [Epoch 01 | Batch 700/1568] Loss: 5.1655 | GradNorm: 2.961
  [Epoch 01 | Batch 750/1568] Loss: 4.8254 | GradNorm: 2.774
  [Epoch 01 | Batch 800/1568] Loss: 6.2326 | GradNorm: 2.

**9. Text Generation**

In [11]:
seed_texts = [
    "the creature stood before me",
    "darkness surrounded the lonely",
    "my heart was filled with",
    "she looked at him with",
    "the storm began to"
]

print("=" * 50)
print("  GENERATED TEXT SAMPLES")
print("=" * 50)

for i, seed in enumerate(seed_texts):
    result = generate_text_gpu(model, seed, next_words=15, temperature=0.8, greedy=True)
    seed_word_count = len(re.findall(r"\w+|[^\w\s]", seed.lower(), re.UNICODE))
    generated = " ".join(result.split()[seed_word_count:])
    generated = generated.replace("< end >", "").replace("<end>", "").strip()
    print(f"Sentence {i+1}: {seed} {generated}")

  GENERATED TEXT SAMPLES
Sentence 1: the creature stood before me .  i had no longer for a few minutes , and my
Sentence 2: darkness surrounded the lonely of a mountain and concealed her extreme tears .  i was unable
Sentence 3: my heart was filled with a manner and the appearance of his own .  i had no
Sentence 4: she looked at him with the heavens .  i was a mere difference of my mind ,
Sentence 5: the storm began to be a few minutes , and i was obliged to sink from my eyes .
